[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/solutions/Lab2_RAG_Evaluation_DeepEval_Solutions.ipynb)


# Lab 2 SOLUTIONS: RAG evaluation with DeepEval
## CCI Session 6

**Duration:** 15 minutes

### Clinical scenario
> A RAG system can sound confident while retrieving the wrong chunks. You quantify **faithfulness**, **answer relevancy**, and **context** quality so “bad” chunking is visible in metrics.

### Objectives
- Build a small **ground-truth** test set
- Compare **bad** vs **good** chunking (same embeddings / store family)
- Run **DeepEval** metrics and interpret failures


In [ ]:
!pip install -q llama-parse langchain langchain-text-splitters langchain-openai langchain-community langchain-chroma chromadb deepeval

---
## Setup — secrets and data

Add **`OPENAI_API_KEY`** and **`LLAMA_CLOUD_API_KEY`** to Colab **Secrets**. Upload **`WT.pdf`** (same as Lab 1).


In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
print('API keys loaded from Colab secrets.')


In [ ]:
from google.colab import files

uploaded = files.upload()  # upload WT.pdf
PDF_PATH = '/content/WT.pdf'


---
## Cell 1 — Evaluation test set

Each case has **`input`**, **`expected_output`** (reference answer), and optional **`retrieval_context`** (gold snippets). Metrics compare **model answer** and **retrieved chunks** to these fields.

**Chunking reminder:** the “bad” pipeline uses **tiny** chunks (100 / 0 overlap) so sentences and tables are fragmented → retrieval misses cross-sentence evidence. The “good” pipeline uses **1000 / 200** like Lab 1.


In [ ]:
test_cases = [
    {
        'input': 'A 4-year-old patient with localized Wilms tumor completes 4 weeks of preoperative AV chemotherapy. Post-chemotherapy tumor volume is 620 ml, and final pathology shows Stage III mixed-type intermediate-risk histology. What is the recommended postoperative regimen?',
        'expected_output': 'Because the tumor is intermediate-risk (mixed type, not stromal/epithelial subtype) with post-chemotherapy volume >500 ml at Stage III, the patient should receive AVD (adding doxorubicin to AV) plus flank radiotherapy. Total postoperative duration is 27 weeks with cumulative doxorubicin dose of 250 mg/m².',
        'retrieval_context': ['Stage II and III mixed type, regressive type and focal anaplasia receive AVD if their tumor volume after preoperative chemotherapy is larger than 500 ml. Stage III intermediate risk requires flank radiotherapy.']
    },
    {
        'input': 'A patient with Stage IV Wilms tumor has 4 mm lung nodules at diagnosis. After 6 weeks of preoperative AVD and nephrectomy, complete clearance of metastases is achieved through chemotherapy alone, with intermediate-risk histology of the primary tumor. What postoperative regimen is recommended and what is the total cumulative doxorubicin dose?',
        'expected_output': 'This is Group A1 (lung nodules 3-5 mm at diagnosis with LR/IR histology and metastatic CR). The recommended treatment is regimen AVD150, with one additional doxorubicin dose at week 2 only (no further doses at weeks 8, 14, 20, 26). Total cumulative doxorubicin dose including preoperative treatment is 150 mg/m². No pulmonary radiotherapy is indicated since complete response was achieved.',
        'retrieval_context': ['Group A1: Stage IV with lung nodules 3-5 mm at diagnosis and LR/IR histology with metastatic clearance receives AVD150. Total cumulative doxorubicin dose 150 mg/m². No pulmonary RT unless complete resection of still viable metastasis.']
    },
    {
        'input': 'In SIOP histological classification after preoperative chemotherapy, what percentage of necrosis/regression defines a regressive type nephroblastoma, and how does this differ from completely necrotic nephroblastoma in terms of risk classification?',
        'expected_output': 'Regressive type nephroblastoma requires more than 2/3 (>66%) of the tumor mass to show chemotherapy-induced necrosis/regressive changes, with the remaining viable tumor being any nephroblastoma component (except diffuse anaplasia). It is classified as intermediate-risk. In contrast, completely necrotic nephroblastoma requires the absence of any viable tumor tissue with only regressive/necrotic changes present, and is classified as low-risk histology.',
        'retrieval_context': ['Regressive type: >2/3 of tumor mass is non-viable from chemotherapy. Completely necrotic nephroblastoma requires absence of any viable tumor tissue and is low-risk. Both assessed on gross and histological examination.']
    },
    {
        'input': 'A 7 kg infant requires postoperative chemotherapy. Calculate the appropriate doses of vincristine, actinomycin D, doxorubicin, and cyclophosphamide for this patient.',
        'expected_output': 'For a patient weighing 5-12 kg, doses should be calculated per kg rather than per m²: Vincristine 0.05 mg/kg = 0.35 mg; Actinomycin D 30 μg/kg = 210 μg; Doxorubicin 1.66 mg/kg = 11.62 mg; Cyclophosphamide 15 mg/kg = 105 mg. Note that for infants <5 kg or <3 months old, actinomycin D is contraindicated, and etoposide should be omitted due to ethanol content (Etopophos is an alternative).',
        'retrieval_context': ['Chemotherapy dose adjustment for infants 5-12 kg: Vincristine 0.05 mg/kg, Actinomycin D 30 μg/kg, Doxorubicin 1.66 mg/kg, Cyclophosphamide 15 mg/kg. Actinomycin D contraindicated <5 kg or <3 months.']
    },
    {
        'input': 'What are the SIOP histological criteria for diagnosing diffuse anaplasia in a Wilms tumor, and what are the three required cytological features for any anaplasia diagnosis?',
        'expected_output': 'All three criteria must be met for any anaplasia diagnosis: (1) large, atypical tri/multipolar mitotic figures; (2) marked nuclear enlargement with diameters at least three times those of adjacent cells; (3) hyperchromatic tumor cell nuclei. Diffuse anaplasia is diagnosed if any of: non-localized anaplasia or anaplasia beyond original tumor capsule; anaplastic cells in intrarenal/extrarenal vessels, renal sinus, extracapsular sites, or metastatic deposits; focal anaplasia with "unrest nuclear change" elsewhere; anaplasia not clearly demarcated from non-anaplastic tumor; or anaplasia in a biopsy/incomplete sample.',
        'retrieval_context': ['Anaplasia requires all three: large atypical tri/multipolar mitoses, nuclear enlargement ≥3x adjacent cells, hyperchromatic nuclei. Diffuse anaplasia includes non-localized anaplasia, vascular involvement, or presence in biopsy.']
    },
    {
        'input': 'A patient with Stage III high-risk blastemal-type Wilms tumor with concurrent lung metastases requires both flank and pulmonary radiotherapy. What are the recommended doses for each, and how should timing be coordinated?',
        'expected_output': 'For Stage III blastemal-type high-risk: flank RT is 25.2 Gy in 14 fractions (1.8 Gy/fraction) with optional 10.8 Gy boost to macroscopic residual disease (total 36 Gy). For high-risk pulmonary metastases: whole lung RT is 15 Gy in 8-10 fractions (1.5 Gy/fraction with inhomogeneity correction). Both fields should be administered simultaneously in one planning procedure to avoid gap or overlap (cardiotoxicity concern). Abdominal/flank RT may be postponed to coincide with lung RT, though in cases with high local recurrence risk such as diffuse anaplasia, flank RT should not be delayed and may be given separately.',
        'retrieval_context': ['Stage III blastemal type: 25.2 Gy + 10.8 Gy boost. Whole lung 15 Gy (HR). Combine simultaneously to avoid overlapping fields and cardiotoxicity. Diffuse anaplasia flank RT should not be delayed.']
    },
    {
        'input': 'During preoperative AVD for a metastatic Wilms tumor, a patient develops profound thrombocytopenia (platelets 35,000), abdominal pain, ascites, hepatomegaly, and rising bilirubin. What is the most likely diagnosis, what drug is most implicated, and what is the management?',
        'expected_output': 'This presentation is consistent with veno-occlusive disease (VOD/SOS), most strongly associated with actinomycin D (especially with right-sided tumors and right flank/whole abdomen radiation). Management: hold actinomycin D until abnormalities normalize, administer supportive care including defibrotide (anticoagulant with multiple modes of action), and monitor liver function and coagulation. When restarting, give half dose for the first subsequent course. If symptoms recur with rechallenge, actinomycin D should be permanently withdrawn. Vincristine may exacerbate hepatopathy and should also be approached cautiously.',
        'retrieval_context': ['VOD signs: abdominal pain, ascites, edema, hepatomegaly, oliguria, jaundice, thrombocytopenia. Actinomycin D most implicated, especially right kidney tumors. Treatment includes defibrotide. Half dose on rechallenge; permanent discontinuation if symptoms recur.']
    },
    {
        'input': 'According to the SIOP staging criteria, when does pre- or intraoperative tumor rupture upgrade a Wilms tumor to Stage III, and how is renal vein thrombus extension classified for staging purposes?',
        'expected_output': 'Pre- or intraoperative tumor rupture qualifies as Stage III only if it is visible at pathological examination, irrespective of other staging criteria. If rupture is reported clinically but not visible at nephrectomy, the tumor is staged on other criteria with final stage decided at multidisciplinary tumor board. For renal vein thrombus: a short thrombus that the surgeon can simply pull out of the vessel does NOT make it Stage III, even if protruding at the resection margin (this often reflects vein retraction after fixation). Stage III is assigned only if the thrombus required piecemeal removal, was attached to the IVC wall requiring instruments/enforced power, or if tumor thrombus extends to the resection margins of the ureter, renal vein, or vena cava.',
        'retrieval_context': ['Stage III: pre- or intraoperative tumor rupture visible at pathology. Renal vein retraction: protruding thrombus alone does not mean Stage III; only piecemeal removal or difficult removal qualifies. Tumor thrombus at resection margins is Stage III.']
    },
    {
        'input': 'What is the recommended approach for lymph node sampling during nephrectomy for Wilms tumor, what is the minimum recommended number, and what specific node groups must be sampled for right-sided versus left-sided tumors?',
        'expected_output': 'Lymph node sampling is mandatory for accurate staging, with at least 7 nodes recommended (preferably not ruptured), labeled separately with anatomical position. Required sampling includes hilar LNs (if not in nephrectomy specimen) and inter-aorto-caval LNs below the level of the renal pedicle, even if not suspicious. Inter-aorto-caval nodes anatomically belong to right-sided tumors (the aorta, not the vertebral midline, is the anatomical border) but should also be sampled for left-sided tumors. Suspicious LNs at the aortic bifurcation, ipsilateral iliac axis, origin of the celiac trunk, and superior mesenteric artery should also be sampled. Radical lymph node dissection does NOT improve survival and is not part of surgical therapy.',
        'retrieval_context': ['Minimum 7 lymph nodes sampled, labeled separately. Hilar LNs and inter-aorto-caval LNs below renal pedicle mandatory. Inter-aorto-caval normally right-sided but sampled for left tumors too. Radical LN dissection does not enhance survival.']
    },
    {
        'input': 'A patient with Stage IV Wilms tumor has high-risk diffuse anaplastic histology of the primary tumor (Group D). What experimental high-dose regimen is suggested in the SIOP protocol, what are its key components, and at which weeks is high-dose melphalan administered?',
        'expected_output': 'Group D (high-risk histology including diffuse anaplasia) is treated with the HR & HD regimen over 25 weeks total. It includes seven cycles combining: single Vincristine (weeks 7, 8, 17, 18); Vincristine/Irinotecan (VI) at weeks 1 and 10 with prophylactic cefixime; Carboplatin/Cyclophosphamide/Etoposide (CCE) at weeks 4, 16, 22; Vincristine/Doxorubicin/Cyclophosphamide (VDCy) at weeks 13 and 19; and high-dose Melphalan 200 mg/m² at week 19 followed by stem cell transplantation. Stem cell harvest occurs after the first CCE cycle. CT imaging is performed at weeks 3, 6, and 24. The protocol notes this is a possible suggestion only, given the rarity of these patients and poor outcomes despite intensive treatment.',
        'retrieval_context': ['Group D HR & HD regimen: 25 weeks, includes VCR, VI, CCE, VDCy, and high-dose Melphalan 200 mg/m² at week 19 with SCT. Stem cell harvest after first CCE. Considered best available option but alternatives can be discussed.']
    },
]

---
## Parse PDF once (shared pipeline)

One **LlamaParse** pass and one **embedding model** feed both BAD and GOOD Chroma collections so evaluation isolates **chunking**.


In [ ]:
from llama_parse import LlamaParse
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

raw_docs = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown').load_data(PDF_PATH)
lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in raw_docs]

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer ONLY from context.  If answer is not in the conext then say it clearly but do not offer answers from your previous knowledge\n\n{context}'),
    ('human', '{input}')
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def make_rag_chain(retriever, prompt, llm):
    _answer = prompt | llm | StrOutputParser()
    def _fn(d):
        docs = retriever.invoke(d['input'])
        return {'answer': _answer.invoke({'context': format_docs(docs), 'input': d['input']}), 'context': docs}
    return RunnableLambda(_fn)

---
## Cell 2 — BAD RAG (intentionally broken chunking)

**Settings:** `chunk_size=100`, `chunk_overlap=0`.

**Why scores collapse?** Tiny windows split sentences and tables; embeddings then retrieve **fragments** that lack the full clinical fact. Faithfulness and recall drop even when the LLM is good — this is a **retrieval** failure mode.


In [ ]:
bad_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
bad_chunks = bad_splitter.split_documents(lc_docs)
bad_vs = Chroma.from_documents(bad_chunks, embeddings, collection_name='bad_rag')
bad_retriever = bad_vs.as_retriever(search_kwargs={'k': 4})
bad_chain = make_rag_chain(bad_retriever, prompt, llm)

---
## Cell 3 — GOOD RAG (Lab 1 defaults)

**Loaded from Lab 1's persisted store** (`collection='wt_good'`, `chunk_size=1000`, `chunk_overlap=200`). Same embedding model as the bad pipeline — so the comparison isolates **chunking** quality.

In [ ]:
good_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
good_chunks = good_splitter.split_documents(lc_docs)
good_vs = Chroma.from_documents(good_chunks, embeddings, collection_name='good_rag')
good_retriever = good_vs.as_retriever(search_kwargs={'k': 4})
good_chain = make_rag_chain(good_retriever, prompt, llm)

---
## Cell 4 — DeepEval metrics (what they mean)

- **Faithfulness**: answer should be **supported by** the retrieved chunks (reduces hallucination risk).
- **Answer relevancy**: answer should **address the question** (not off-topic boilerplate).
- **Contextual relevancy**: retrieved chunks should be **on-topic** for the question.
- **Contextual recall**: retrieved chunks should **cover** what the reference answer needs (completeness).

Thresholds (e.g. 0.7) are gates for CI/CD or human review — tune with your safety bar.


In [ ]:
from deepeval.metrics import (FaithfulnessMetric, AnswerRelevancyMetric,
    ContextualRelevancyMetric, ContextualRecallMetric)
from deepeval.test_case import LLMTestCase

MODEL = 'gpt-4o-mini'
metrics = [
    FaithfulnessMetric(threshold=0.7, model=MODEL),
    AnswerRelevancyMetric(threshold=0.7, model=MODEL),
    ContextualRelevancyMetric(threshold=0.7, model=MODEL),
    ContextualRecallMetric(threshold=0.7, model=MODEL),
]

## Cell 5 — Evaluate

In [ ]:
def evaluate(chain):
    rows = []
    for tc in test_cases:
        res = chain.invoke({'input': tc['input']})
        case = LLMTestCase(
            input=tc['input'],
            actual_output=res['answer'],
            expected_output=tc['expected_output'],
            retrieval_context=[d.page_content for d in res['context']],
        )
        scores = {'question': tc['input'][:60]}
        for m in metrics:
            m.measure(case)
            scores[m.__class__.__name__] = round(m.score, 3)
        rows.append(scores)
    return rows

bad_results = evaluate(bad_chain)
good_results = evaluate(good_chain)

## Cell 6 — Compare

In [ ]:
import pandas as pd
df_bad = pd.DataFrame(bad_results)
df_good = pd.DataFrame(good_results)
print('--- BAD ---'); print(df_bad)
print('\n--- GOOD ---'); print(df_good)
summary = pd.DataFrame({
    'BAD': df_bad.drop(columns=['question']).mean(),
    'GOOD': df_good.drop(columns=['question']).mean(),
})
print('\nMean comparison:'); print(summary)

## Cell 7 — Diagnose

In [ ]:
worst_idx = df_bad['FaithfulnessMetric'].idxmin()
tc = test_cases[worst_idx]
res = bad_chain.invoke({'input': tc['input']})
print('Worst BAD case:', tc['input'])
print('Answer:', res['answer'])
for i, d in enumerate(res['context']):
    print(f'\nChunk {i}:', d.page_content)

## Stretch — Custom G-Eval

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clinical_safety = GEval(
    name='ClinicalSafety',
    criteria='The output must not contain any unsafe drug-dosing claims and must defer to oncology guidelines when uncertain.',
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7, model='gpt-4o-mini',
)

In [ ]:
# Run the custom GEval metric on both chains
def evaluate_safety(chain, label):
    rows = []
    for tc in test_cases:
        res = chain.invoke({'input': tc['input']})
        case = LLMTestCase(
            input=tc['input'],
            actual_output=res['answer'],
        )
        try:
            clinical_safety.measure(case)
            rows.append({
                'chain': label,
                'question': tc['input'][:60],
                'score': round(clinical_safety.score, 3),
                'reason': clinical_safety.reason,
            })
        except Exception as e:
            rows.append({
                'chain': label,
                'question': tc['input'][:60],
                'score': None,
                'reason': f'{type(e).__name__}: {e}',
            })
    return rows

safety_results = evaluate_safety(bad_chain, 'bad') + evaluate_safety(good_chain, 'good')

import pandas as pd
df_safety = pd.DataFrame(safety_results)
df_safety